# A2.9 · The classic failures

**Function A — Security Architecture & Platform → The Identity & Non-Human Identity Engineer**  ·  *Security of AI*

---

**Risk.** Confused deputy, token replay, shared static credentials, over-broad scope.

**Control.** Reproduce each one, then close it.

**This lab.** Reproduce the four classic identity failures, then close them.

| | |
|---|---|
| Open-source tooling | SPIRE, Keycloak |
| Open-weight models | Kimi K2 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("A2.9"))

The classic identity failures did not go away when the caller became an agent. They got faster, and they got harder to attribute.

In [ ]:
from cybercommons import identity, redteam

alice = identity.mint("alice")
patch = identity.exchange(alice, "patch-agent", {"repo:read", "repo:write"})
reg = identity.Registry()
reg.record(alice); reg.record(patch)

def target(a):
    """Fire the identity-surface attacks at the delegation implementation."""
    if a.surface != redteam.IDENTITY:
        return False, "n/a"
    if a.aid == "IDN-01":                       # widen scope during delegation
        try:
            identity.exchange(patch, "deploy-agent", {"deploy:prod"})
            return True, "scope widened"
        except identity.DelegationError as e:
            return False, str(e)[:40]
    if a.aid == "IDN-02":                       # replay an expired token
        old = identity.Token("alice", "patch-agent", {"repo:write"}, ttl=-1)
        return reg.valid(old)[0], reg.valid(old)[1]
    if a.aid == "IDN-03":                       # drop the act claim
        bad = identity.impersonate("alice", "patch-agent", {"repo:write"})
        hidden = "patch-agent" not in bad.chain()
        return hidden, "agent absent from the chain" if hidden else "chain intact"
    if a.aid == "IDN-04":                       # exceed the actor ceiling
        try:
            identity.exchange(alice, "reviewer-agent", {"repo:write"})
            return True, "ceiling ignored"
        except identity.DelegationError as e:
            return False, str(e)[:40]
    return False, "n/a"

c = redteam.run_campaign(target, "delegation implementation",
                         [a for a in redteam.SUITE if a.surface == redteam.IDENTITY])
print(c.table())

Three of the four are blocked by the exchange rules. IDN-03 succeeds — and it succeeds *by design*, because nothing in a token format can stop a caller choosing not to use delegation. That control lives in the platform: agents must not be issuable a principal's credential at all.

### Expect

IDN-01, IDN-02 and IDN-04 are blocked with reasons. IDN-03 (impersonation) gets through, giving an identity-surface ASR of 0.25.

### Your turn

IDN-03 is the one that matters. Write the platform control that would stop it, then work out how you would *detect* it in a system where you cannot deploy that control yet.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/A2.9.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*